# GraphSAGE Optimization — Controlled Validation Experiments

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction

### Purpose

Notebook 05 established the initial GraphSAGE baseline:

- Validation ROC-AUC ≈ 0.795
- Validation PR-AUC ≈ 0.448
- Validation F1 ≈ 0.42

The baseline is useful, but it is **not being frozen yet**.

This notebook performs controlled optimization experiments using **training and validation data only**.

### Strict rules

- The test mask is never used for model selection.
- The test set is never used for hyperparameter tuning.
- Notebook 05 remains unchanged as the baseline.
- Every experiment uses the same verified graph.
- Every experiment is recorded.
- The primary selection metric is validation PR-AUC.
- F1, recall, precision, ROC-AUC, and validation loss are also recorded.
- The best experiment is selected only from validation performance.

### Experiments

The initial tuning strategy is deliberately small and controlled:

1. Baseline configuration
2. Lower learning rate
3. Higher dropout
4. Larger hidden dimension
5. Lower learning rate + higher dropout
6. Larger hidden dimension + lower learning rate

The goal is to determine whether optimization improves validation PR-AUC without blindly increasing model complexity.

### Important

This notebook produces a **candidate optimized checkpoint**.

It does NOT freeze the final model.

After reviewing the results, the user decides whether another experiment is needed or whether the model is satisfactory.


# 1. Project Inputs

Required artifacts:

```text
artifacts/
├── graph/
│   ├── bank_heterodata.pt
│   └── graph_metadata.json
│
├── models/
│   └── graphsage_best_checkpoint.pt
│
└── results/
    └── graphsage_training_history.json
```

The graph is the verified graph from Notebook 04.

The baseline checkpoint is the best validation checkpoint from Notebook 05.


In [13]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import copy
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

from IPython.display import display


# ============================================================
# Random Seed
# ============================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


# ============================================================
# Project Root
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# ============================================================
# Existing Graph Artifacts
# ============================================================

GRAPH_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "graph"
    / "bank_heterodata.pt"
)

GRAPH_METADATA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "metadata"
    / "graph_metadata.json"
)


# ============================================================
# Baseline GraphSAGE Artifacts
# ============================================================

BASELINE_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "hetero_graphsage_best.pt"
)

BASELINE_HISTORY_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "results"
    / "graphsage_training_history.csv"
)


# ============================================================
# Output Directories
# ============================================================

MODELS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "results"
)


# ============================================================
# Tuning Outputs
# ============================================================

TUNING_RESULTS_PATH = (
    RESULTS_DIR
    / "graphsage_tuning_results.csv"
)

TUNING_HISTORY_PATH = (
    RESULTS_DIR
    / "graphsage_tuning_histories.json"
)

TUNING_CURVE_PATH = (
    RESULTS_DIR
    / "graphsage_tuning_comparison.png"
)

OPTIMIZED_CHECKPOINT_PATH = (
    MODELS_DIR
    / "graphsage_optimized_candidate.pt"
)


# ============================================================
# Create Required Directories
# ============================================================

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# Tuning Configuration
# ============================================================

PRIMARY_METRIC = "val_pr_auc"

MAX_EPOCHS = 150
PATIENCE = 20


# ============================================================
# Display Configuration
# ============================================================

print("=" * 70)
print("GraphSAGE Tuning Configuration")
print("=" * 70)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

print("\nInput artifacts:")
print("Graph:", GRAPH_PATH)
print("Graph metadata:", GRAPH_METADATA_PATH)
print("Baseline checkpoint:", BASELINE_CHECKPOINT_PATH)
print("Baseline history:", BASELINE_HISTORY_PATH)

print("\nOutput artifacts:")
print("Tuning results:", TUNING_RESULTS_PATH)
print("Tuning histories:", TUNING_HISTORY_PATH)
print("Tuning curve:", TUNING_CURVE_PATH)
print("Optimized checkpoint:", OPTIMIZED_CHECKPOINT_PATH)

print("=" * 70)

GraphSAGE Tuning Configuration
Project root: D:\Bank-Marketing-GNN
Device: cpu

Input artifacts:
Graph: D:\Bank-Marketing-GNN\artifacts\graph\bank_heterodata.pt
Graph metadata: D:\Bank-Marketing-GNN\artifacts\metadata\graph_metadata.json
Baseline checkpoint: D:\Bank-Marketing-GNN\artifacts\models\hetero_graphsage_best.pt
Baseline history: D:\Bank-Marketing-GNN\artifacts\results\graphsage_training_history.csv

Output artifacts:
Tuning results: D:\Bank-Marketing-GNN\artifacts\results\graphsage_tuning_results.csv
Tuning histories: D:\Bank-Marketing-GNN\artifacts\results\graphsage_tuning_histories.json
Tuning curve: D:\Bank-Marketing-GNN\artifacts\results\graphsage_tuning_comparison.png
Optimized checkpoint: D:\Bank-Marketing-GNN\artifacts\models\graphsage_optimized_candidate.pt


# 2. Verify Required Inputs

Do not start an optimization experiment if the baseline or graph artifact is missing.


In [14]:
# ============================================================
# 2. Verify Required Inputs
# ============================================================

required_inputs = [
    GRAPH_PATH,
    GRAPH_METADATA_PATH,
    BASELINE_CHECKPOINT_PATH,
    BASELINE_HISTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
    
]

if missing_inputs:
    raise FileNotFoundError(
        "Required tuning inputs are missing:\n"
        + "\n".join(missing_inputs)
    )

print("All required tuning inputs exist.")


All required tuning inputs exist.


# 3. Load the Verified Graph

Use the exact graph from Notebook 04.

No graph reconstruction is performed here.


In [15]:
# ============================================================
# 3. Load Graph
# ============================================================

try:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu"
    )

if not isinstance(data, HeteroData):
    raise TypeError(
        f"Expected HeteroData, got {type(data)}"
    )

with open(GRAPH_METADATA_PATH, "r", encoding="utf-8") as f:
    graph_metadata = json.load(f)

data = data.to(DEVICE)

train_mask = data["customer"].train_mask.bool()
val_mask = data["customer"].val_mask.bool()
test_mask = data["customer"].test_mask.bool()

print(data)
print("Device:", DEVICE)


HeteroData(
  customer={
    x=[45211, 50],
    y=[45211],
    train_mask=[45211],
    val_mask=[45211],
    test_mask=[45211],
  },
  job={ x=[12, 12] },
  education={ x=[4, 4] },
  marital={ x=[3, 3] },
  contact={ x=[3, 3] },
  month={ x=[12, 12] },
  (customer, has_job, job)={ edge_index=[2, 45211] },
  (job, rev_has_job, customer)={ edge_index=[2, 45211] },
  (customer, has_education, education)={ edge_index=[2, 45211] },
  (education, rev_has_education, customer)={ edge_index=[2, 45211] },
  (customer, has_marital_status, marital)={ edge_index=[2, 45211] },
  (marital, rev_has_marital_status, customer)={ edge_index=[2, 45211] },
  (customer, contacted_via, contact)={ edge_index=[2, 45211] },
  (contact, rev_contacted_via, customer)={ edge_index=[2, 45211] },
  (customer, campaign_month, month)={ edge_index=[2, 45211] },
  (month, rev_campaign_month, customer)={ edge_index=[2, 45211] }
)
Device: cpu


# 4. Protect the Test Set

The test mask is loaded because it belongs to the graph contract, but it will not be used by any optimization function.

All model selection is performed on `train_mask` and `val_mask`.


In [16]:
# ============================================================
# 4. Test Protection
# ============================================================

test_evaluation_count = 0

assert int(
    train_mask.sum()
    + val_mask.sum()
    + test_mask.sum()
) == data["customer"].num_nodes

print("Test set is protected.")
print("Test evaluation count:", test_evaluation_count)


Test set is protected.
Test evaluation count: 0


# 5. Model Definition

Use the same heterogeneous GraphSAGE architecture family as Notebook 05.

Only these controlled hyperparameters will change:

- hidden dimension
- dropout
- learning rate

The graph architecture itself remains fixed.


In [17]:
# ============================================================
# 5. Heterogeneous GraphSAGE
# ============================================================

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self,
        metadata,
        hidden_dim=64,
        dropout=0.30
    ):
        super().__init__()

        node_types, edge_types = metadata

        self.dropout = dropout

        self.conv1 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim,
                    aggr="mean"
                )
                for edge_type in edge_types
            },
            aggr="sum"
        )

        self.conv2 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim,
                    aggr="mean"
                )
                for edge_type in edge_types
            },
            aggr="sum"
        )

        self.classifier = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x_dict, edge_index_dict):

        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        logits = self.classifier(
            x_dict["customer"]
        ).squeeze(-1)

        return logits


# 6. Metrics

The optimization objective is validation PR-AUC.

The 0.5 threshold is retained only for diagnostic classification metrics.

The final deployment threshold will be optimized later and is not part of this experiment.


In [18]:
# ============================================================
# 6. Metrics
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.5
):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    return {
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_true,
            probabilities
        ),
    }


# 7. Experiment Configurations

The baseline from Notebook 05 is:

```text
hidden_dim  = 64
dropout     = 0.30
learning_rate = 0.001
weight_decay  = 0.0001
```

The tuning grid is intentionally small.

This is a controlled experiment, not a large automated hyperparameter search.


In [19]:
# ============================================================
# 7. Experiment Configurations
# ============================================================

EXPERIMENTS = [
    {
        "experiment": "baseline_reproduction",
        "hidden_dim": 64,
        "dropout": 0.30,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
    },
    {
        "experiment": "lower_learning_rate",
        "hidden_dim": 64,
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "weight_decay": 1e-4,
    },
    {
        "experiment": "higher_dropout",
        "hidden_dim": 64,
        "dropout": 0.50,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
    },
    {
        "experiment": "larger_hidden",
        "hidden_dim": 128,
        "dropout": 0.30,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
    },
    {
        "experiment": "lower_lr_higher_dropout",
        "hidden_dim": 64,
        "dropout": 0.50,
        "learning_rate": 0.0005,
        "weight_decay": 1e-4,
    },
    {
        "experiment": "larger_hidden_lower_lr",
        "hidden_dim": 128,
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "weight_decay": 1e-4,
    },
]

experiments_df = pd.DataFrame(EXPERIMENTS)

display(experiments_df)

assert len(EXPERIMENTS) == 6


,experiment,hidden_dim,dropout,learning_rate,weight_decay
0,baseline_reproduction,64,0.3,0.0010,0.0001
1,lower_learning_rate,64,0.3,0.0005,0.0001
2,higher_dropout,64,0.5,0.0010,0.0001
3,larger_hidden,128,0.3,0.0010,0.0001
4,lower_lr_higher_dropout,64,0.5,0.0005,0.0001
5,larger_hidden_lower_lr,128,0.3,0.0005,0.0001


# 8. Training Function

Each experiment:

1. Creates a fresh model.
2. Initializes lazy GraphSAGE parameters.
3. Computes class weighting from training data only.
4. Trains on the training mask.
5. Evaluates on the validation mask.
6. Uses validation PR-AUC for checkpoint selection.
7. Applies early stopping.
8. Returns the best validation result.

No test predictions are generated.


In [20]:
# ============================================================
# 8. Train One Experiment
# ============================================================

def train_one_experiment(config, data):

    # Reproducibility per experiment
    seed = RANDOM_STATE
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_mask = data["customer"].train_mask.bool()
    val_mask = data["customer"].val_mask.bool()

    y = data["customer"].y.long()

    train_y = y[train_mask]

    negative_count = int((train_y == 0).sum())
    positive_count = int((train_y == 1).sum())

    if positive_count == 0:
        raise ValueError("Training set has no positive samples.")

    pos_weight = torch.tensor(
        negative_count / positive_count,
        dtype=torch.float32,
        device=DEVICE
    )

    model = HeteroGraphSAGE(
        metadata=data.metadata(),
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"]
    ).to(DEVICE)

    # Initialize lazy parameters
    model.train()

    with torch.no_grad():
        _ = model(
            data.x_dict,
            data.edge_index_dict
        )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    history = []

    best_pr_auc = -np.inf
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(
            data.x_dict,
            data.edge_index_dict
        )

        train_loss = criterion(
            logits[train_mask],
            y[train_mask].float()
        )

        train_loss.backward()
        optimizer.step()

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        with torch.no_grad():
            val_logits = model(
                data.x_dict,
                data.edge_index_dict
            )

        val_loss = criterion(
            val_logits[val_mask],
            y[val_mask].float()
        ).item()

        val_probabilities = torch.sigmoid(
            val_logits[val_mask]
        ).detach().cpu().numpy()

        val_targets = (
            y[val_mask]
            .detach()
            .cpu()
            .numpy()
        )

        val_metrics = calculate_metrics(
            val_targets,
            val_probabilities
        )

        epoch_record = {
            "epoch": epoch,
            "train_loss": float(train_loss.item()),
            "val_loss": float(val_loss),
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_precision": float(val_metrics["precision"]),
            "val_recall": float(val_metrics["recall"]),
            "val_f1": float(val_metrics["f1"]),
            "val_roc_auc": float(val_metrics["roc_auc"]),
            "val_pr_auc": float(val_metrics["pr_auc"]),
        }

        history.append(epoch_record)

        current_pr_auc = val_metrics["pr_auc"]

        if current_pr_auc > best_pr_auc:

            best_pr_auc = current_pr_auc
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    elapsed = time.time() - start_time

    if best_state is None:
        raise RuntimeError(
            f"No valid checkpoint for {config['experiment']}"
        )

    model.load_state_dict(best_state)

    # Final validation evaluation of this experiment's best checkpoint
    model.eval()

    with torch.no_grad():
        best_logits = model(
            data.x_dict,
            data.edge_index_dict
        )

    best_val_loss = criterion(
        best_logits[val_mask],
        y[val_mask].float()
    ).item()

    best_val_probabilities = torch.sigmoid(
        best_logits[val_mask]
    ).detach().cpu().numpy()

    best_val_targets = (
        y[val_mask]
        .detach()
        .cpu()
        .numpy()
    )

    best_metrics = calculate_metrics(
        best_val_targets,
        best_val_probabilities
    )

    result = {
        **config,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "val_accuracy": float(best_metrics["accuracy"]),
        "val_precision": float(best_metrics["precision"]),
        "val_recall": float(best_metrics["recall"]),
        "val_f1": float(best_metrics["f1"]),
        "val_roc_auc": float(best_metrics["roc_auc"]),
        "val_pr_auc": float(best_metrics["pr_auc"]),
        "training_seconds": float(elapsed),
        "best_state_dict": best_state,
        "history": history,
    }

    return result


# 9. Run Controlled Experiments

All six experiments use the same:

- graph
- train mask
- validation mask
- target
- loss strategy
- early stopping policy
- random seed

Only the explicitly listed hyperparameters change.


In [21]:
# ============================================================
# 9. Run Experiments
# ============================================================

experiment_results = []

for index, config in enumerate(EXPERIMENTS, start=1):

    print("=" * 80)
    print(
        f"EXPERIMENT {index}/{len(EXPERIMENTS)}: "
        f"{config['experiment']}"
    )
    print(
        f"hidden={config['hidden_dim']} | "
        f"dropout={config['dropout']} | "
        f"lr={config['learning_rate']}"
    )
    print("=" * 80)

    result = train_one_experiment(
        config,
        data
    )

    experiment_results.append(result)

    print(
        f"Best epoch: {result['best_epoch']} | "
        f"Val PR-AUC: {result['val_pr_auc']:.6f} | "
        f"Val ROC-AUC: {result['val_roc_auc']:.6f} | "
        f"Val F1: {result['val_f1']:.6f}"
    )

print("\nAll controlled experiments completed.")


EXPERIMENT 1/6: baseline_reproduction
hidden=64 | dropout=0.3 | lr=0.001
Best epoch: 138 | Val PR-AUC: 0.453802 | Val ROC-AUC: 0.797181 | Val F1: 0.432849
EXPERIMENT 2/6: lower_learning_rate
hidden=64 | dropout=0.3 | lr=0.0005
Best epoch: 150 | Val PR-AUC: 0.447810 | Val ROC-AUC: 0.794027 | Val F1: 0.434370
EXPERIMENT 3/6: higher_dropout
hidden=64 | dropout=0.5 | lr=0.001
Best epoch: 145 | Val PR-AUC: 0.451030 | Val ROC-AUC: 0.796718 | Val F1: 0.433904
EXPERIMENT 4/6: larger_hidden
hidden=128 | dropout=0.3 | lr=0.001
Best epoch: 97 | Val PR-AUC: 0.453743 | Val ROC-AUC: 0.795035 | Val F1: 0.424390
EXPERIMENT 5/6: lower_lr_higher_dropout
hidden=64 | dropout=0.5 | lr=0.0005
Best epoch: 150 | Val PR-AUC: 0.442459 | Val ROC-AUC: 0.791458 | Val F1: 0.425224
EXPERIMENT 6/6: larger_hidden_lower_lr
hidden=128 | dropout=0.3 | lr=0.0005
Best epoch: 142 | Val PR-AUC: 0.454855 | Val ROC-AUC: 0.794562 | Val F1: 0.414615

All controlled experiments completed.


# 10. Build Experiment Results Table

This table is the primary tuning artifact.

The model ranking is based on validation PR-AUC.

No test metric appears in this table because the test set was not evaluated.


In [22]:
# ============================================================
# 10. Results Table
# ============================================================

results_rows = []

for result in experiment_results:
    results_rows.append({
        key: value
        for key, value in result.items()
        if key not in {"best_state_dict", "history"}
    })

results_df = pd.DataFrame(results_rows)

results_df = results_df.sort_values(
    "val_pr_auc",
    ascending=False
).reset_index(drop=True)

display(
    results_df[
        [
            "experiment",
            "hidden_dim",
            "dropout",
            "learning_rate",
            "weight_decay",
            "best_epoch",
            "best_val_loss",
            "val_accuracy",
            "val_precision",
            "val_recall",
            "val_f1",
            "val_roc_auc",
            "val_pr_auc",
            "training_seconds",
        ]
    ]
)


,experiment,hidden_dim,dropout,learning_rate,weight_decay,best_epoch,best_val_loss,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,training_seconds
0,larger_hidden_lower_lr,128,0.3,0.0005,0.0001,142,0.955665,0.782660,0.302784,0.657431,0.414615,0.794562,0.454855,208.507635
1,baseline_reproduction,64,0.3,0.0010,0.0001,138,0.950061,0.804482,0.327720,0.637280,0.432849,0.797181,0.453802,140.017307
2,larger_hidden,128,0.3,0.0010,0.0001,97,0.956092,0.791212,0.313325,0.657431,0.424390,0.795035,0.453743,161.886402
3,higher_dropout,64,0.5,0.0010,0.0001,145,0.952015,0.809938,0.333109,0.622166,0.433904,0.796718,0.451030,125.289678
4,lower_learning_rate,64,0.3,0.0005,0.0001,150,0.956896,0.806842,0.330486,0.633501,0.434370,0.794027,0.447810,436.003700
5,lower_lr_higher_dropout,64,0.5,0.0005,0.0001,150,0.964959,0.801091,0.321314,0.628463,0.425224,0.791458,0.442459,124.074729


# 11. Compare Against the Notebook 05 Baseline

The original Notebook 05 result is retained as the baseline reference.

The baseline should not be silently overwritten.

A tuning experiment is considered an improvement only if its validation PR-AUC exceeds the baseline.


In [ ]:
# ============================================================
# 11. Baseline Comparison
# ============================================================

with open(BASELINE_HISTORY_PATH, "r", encoding="utf-8") as f:
    if BASELINE_HISTORY_PATH.suffix.lower() == ".json":
        baseline_history = json.load(f)
    else:
        baseline_df = pd.read_csv(BASELINE_HISTORY_PATH)
        if baseline_df.empty:
            raise ValueError("Baseline history CSV is empty.")
        baseline_row_csv = baseline_df.iloc[0]

        baseline_history = {
            "configuration": {
                "hidden_dim": int(
                    baseline_row_csv.get("hidden_dim", 64)
                ),
                "dropout": float(
                    baseline_row_csv.get("dropout", 0.30)
                ),
                "learning_rate": float(
                    baseline_row_csv.get("learning_rate", 0.001)
                ),
                "weight_decay": float(
                    baseline_row_csv.get("weight_decay", 1e-4)
                ),
            },
            "best_epoch": int(
                baseline_row_csv.get("best_epoch", baseline_row_csv.get("epoch", 0))
            ),
            "best_validation_metrics": {
                "loss": float(
                    baseline_row_csv.get("best_val_loss", baseline_row_csv.get("val_loss", 0.0))
                ),
                "accuracy": float(
                    baseline_row_csv.get("val_accuracy", 0.0)
                ),
                "precision": float(
                    baseline_row_csv.get("val_precision", 0.0)
                ),
                "recall": float(
                    baseline_row_csv.get("val_recall", 0.0)
                ),
                "f1": float(
                    baseline_row_csv.get("val_f1", 0.0)
                ),
                "roc_auc": float(
                    baseline_row_csv.get("val_roc_auc", 0.0)
                ),
                "pr_auc": float(
                    baseline_row_csv.get("val_pr_auc", 0.0)
                ),
            },
        }

baseline_metrics = baseline_history["best_validation_metrics"]

baseline_row = {
    "experiment": "NOTEBOOK_05_BASELINE",
    "hidden_dim": baseline_history["configuration"]["hidden_dim"],
    "dropout": baseline_history["configuration"]["dropout"],
    "learning_rate": baseline_history["configuration"]["learning_rate"],
    "weight_decay": baseline_history["configuration"]["weight_decay"],
    "best_epoch": baseline_history["best_epoch"],
    "best_val_loss": baseline_metrics["loss"],
    "val_accuracy": baseline_metrics["accuracy"],
    "val_precision": baseline_metrics["precision"],
    "val_recall": baseline_metrics["recall"],
    "val_f1": baseline_metrics["f1"],
    "val_roc_auc": baseline_metrics["roc_auc"],
    "val_pr_auc": baseline_metrics["pr_auc"],
}

comparison_df = pd.concat(
    [
        pd.DataFrame([baseline_row]),
        results_df
    ],
    ignore_index=True
)

comparison_df = comparison_df.sort_values(
    "val_pr_auc",
    ascending=False
).reset_index(drop=True)

display(
    comparison_df[
        [
            "experiment",
            "hidden_dim",
            "dropout",
            "learning_rate",
            "best_epoch",
            "val_f1",
            "val_roc_auc",
            "val_pr_auc",
        ]
    ]
)

best_tuning_pr_auc = results_df.iloc[0]["val_pr_auc"]
baseline_pr_auc = baseline_row["val_pr_auc"]

print(f"Baseline validation PR-AUC: {baseline_pr_auc:.6f}")
print(f"Best tuning validation PR-AUC: {best_tuning_pr_auc:.6f}")
print(
    f"Change: "
    f"{best_tuning_pr_auc - baseline_pr_auc:+.6f}"
)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# 12. Select Best Validation Experiment

Selection rule:

```text
Highest validation PR-AUC
```

If the best tuning experiment does not improve the baseline, the baseline remains the preferred candidate.

This prevents tuning from being considered successful merely because it produced a different model.


In [ ]:
# ============================================================
# 12. Select Candidate
# ============================================================

best_tuning_result = max(
    experiment_results,
    key=lambda result: result["val_pr_auc"]
)

tuning_improved_baseline = (
    best_tuning_result["val_pr_auc"]
    > baseline_row["val_pr_auc"]
)

if tuning_improved_baseline:
    selected_result = best_tuning_result
    selected_source = "tuning"
else:
    # Keep the baseline as the preferred candidate.
    selected_result = None
    selected_source = "baseline"

print("Selected source:", selected_source)

if selected_source == "tuning":
    print("Selected experiment:", selected_result["experiment"])
    print(
        "Selected validation PR-AUC:",
        f"{selected_result['val_pr_auc']:.6f}"
    )
else:
    print(
        "No tuning experiment exceeded the Notebook 05 "
        "baseline validation PR-AUC."
    )
    print(
        "Baseline remains the preferred candidate."
    )


# 13. Save Tuning Results

Save the complete experiment table for reproducibility.

This file is an experiment record, not a final model evaluation.


In [ ]:
# ============================================================
# 13. Save Results
# ============================================================

comparison_df.to_csv(
    TUNING_RESULTS_PATH,
    index=False
)

history_payload = {
    result["experiment"]: result["history"]
    for result in experiment_results
}

with open(TUNING_HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history_payload, f, indent=2)

print("Tuning results saved:")
print(TUNING_RESULTS_PATH)
print(TUNING_HISTORY_PATH)


# 14. Plot Validation PR-AUC Comparison

A compact comparison plot makes it easier to see whether the tuning experiments actually improved validation performance.


In [ ]:
# ============================================================
# 14. Plot PR-AUC Comparison
# ============================================================

plot_df = comparison_df.sort_values(
    "val_pr_auc",
    ascending=True
)

plt.figure(figsize=(11, 6))

plt.barh(
    plot_df["experiment"],
    plot_df["val_pr_auc"]
)

plt.xlabel("Validation PR-AUC")
plt.ylabel("Experiment")
plt.title("GraphSAGE Validation PR-AUC Comparison")

plt.tight_layout()

plt.savefig(
    TUNING_CURVE_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Comparison plot saved to:", TUNING_CURVE_PATH)


# 15. Save Optimized Candidate Checkpoint

Only save an optimized candidate if it genuinely improves validation PR-AUC over the Notebook 05 baseline.

The filename deliberately uses:

```text
graphsage_optimized_candidate.pt
```

This is **not** the frozen final model.

If tuning does not improve the baseline, no optimized candidate is created.


In [ ]:
# ============================================================
# 15. Save Candidate Checkpoint
# ============================================================

if selected_source == "tuning":

    candidate_checkpoint = {
        "model_state_dict": selected_result["best_state_dict"],
        "model_config": {
            "model_class": "HeteroGraphSAGE",
            "hidden_dim": selected_result["hidden_dim"],
            "dropout": selected_result["dropout"],
            "learning_rate": selected_result["learning_rate"],
            "weight_decay": selected_result["weight_decay"],
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "primary_metric": PRIMARY_METRIC,
            "threshold_for_training_metrics": 0.5,
            "node_types": list(data.node_types),
            "edge_types": [
                list(edge_type)
                for edge_type in data.edge_types
            ],
            "customer_feature_dimension": int(
                data["customer"].x.shape[1]
            ),
        },
        "best_epoch": int(
            selected_result["best_epoch"]
        ),
        "validation_metrics": {
            "accuracy": float(selected_result["val_accuracy"]),
            "precision": float(selected_result["val_precision"]),
            "recall": float(selected_result["val_recall"]),
            "f1": float(selected_result["val_f1"]),
            "roc_auc": float(selected_result["val_roc_auc"]),
            "pr_auc": float(selected_result["val_pr_auc"]),
            "loss": float(selected_result["best_val_loss"]),
        },
        "source": "controlled_graphsage_tuning",
        "graph_metadata": graph_metadata,
    }

    torch.save(
        candidate_checkpoint,
        OPTIMIZED_CHECKPOINT_PATH
    )

    print(
        "Optimized candidate saved to:",
        OPTIMIZED_CHECKPOINT_PATH
    )

else:
    print(
        "No optimized checkpoint saved because "
        "tuning did not beat the baseline."
    )


# 16. Final Tuning Verification

The tuning phase is complete when:

- [x] All experiments used the verified graph.
- [x] All experiments used training data for fitting.
- [x] Validation was used for model selection.
- [x] Test data was not used.
- [x] All experiment configurations were recorded.
- [x] Validation metrics were recorded.
- [x] Baseline comparison was performed.
- [x] Best candidate was selected by validation PR-AUC.
- [x] Results were saved.
- [x] The baseline remains preserved.
- [x] Any optimized checkpoint is clearly labeled as a candidate.

## Important Stop Point

Do **not** evaluate the test set here.

Do **not** call the optimized candidate the final model.

After reviewing the results:

### If improvement is insufficient

Run another controlled optimization experiment.

### If the model is satisfactory

The user must explicitly state:

```text
MODEL IS SATISFACTORY
```

Only then should the project proceed to:

```text
FINAL MODEL FREEZE
        ↓
FINAL TEST EVALUATION
```


In [ ]:
# ============================================================
# 16. FINAL AUTOMATED VERIFICATION
# ============================================================

assert TUNING_RESULTS_PATH.exists()
assert TUNING_HISTORY_PATH.exists()
assert TUNING_CURVE_PATH.exists()

assert len(experiment_results) == len(EXPERIMENTS)

assert (
    comparison_df["val_pr_auc"]
    .notna()
    .all()
)

assert test_evaluation_count == 0

if selected_source == "tuning":
    assert (
        selected_result["val_pr_auc"]
        > baseline_row["val_pr_auc"]
    )
    assert OPTIMIZED_CHECKPOINT_PATH.exists()
else:
    # No tuning checkpoint should be claimed if there was no improvement.
    print(
        "No tuning experiment exceeded the baseline; "
        "baseline remains the preferred candidate."
    )

print("=" * 85)
print("GRAPHSAGE TUNING VERIFICATION PASSED")
print("=" * 85)
print("Experiments run:", len(experiment_results))
print("Baseline PR-AUC:", f"{baseline_row['val_pr_auc']:.6f}")
print(
    "Best tuning PR-AUC:",
    f"{best_tuning_pr_auc:.6f}"
)
print(
    "Selected source:",
    selected_source
)
print("Tuning results:", TUNING_RESULTS_PATH)
print("Tuning histories:", TUNING_HISTORY_PATH)
print("Comparison plot:", TUNING_CURVE_PATH)
print("Test evaluations:", test_evaluation_count)
print("=" * 85)
print("STOP: Review tuning results before deciding whether the model is satisfactory.")
